# Solution C2 · Wrong labels: where k = 1 collapses and large k holds up

**Type**: Break the model

## Objective

Corrupt a growing share of the training labels (0%, 10%, 20%, 30%, 40%) and measure, for each level, the test accuracy as a function of `k`: the KNN with `k = 1` **memorizes** the errors, while the best `k` grows with the noise.
Check that cross-validation on the noisy training set still finds a `k` that works.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_moons
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier

X, y = make_moons(2000, noise=0.15, random_state=1)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.5, random_state=0)
Ks = [1, 3, 5, 9, 15, 25, 41, 61, 91, 121]
ps = [0, 0.1, 0.2, 0.3, 0.4]


def noisy(p):
    """Training labels with a fraction p swapped at random (the test set stays clean)."""
    swap = np.random.default_rng(0).random(len(y_tr)) < p
    return np.where(swap, 1 - y_tr, y_tr)

## Commented solution

In [ ]:
acc = pd.DataFrame(index=ps, columns=Ks, dtype=float)
for p in ps:
    y_noisy = noisy(p)
    for k in Ks:
        acc.loc[p, k] = KNeighborsClassifier(k).fit(X_tr, y_noisy).score(X_te, y_te)          # noisy training labels, clean test set
acc.round(3)

In [ ]:
optimal_k = acc.idxmax(axis=1)
optimal_acc = acc.max(axis=1)
k1_acc = acc[1]
k1_train_acc = pd.Series({p: KNeighborsClassifier(1).fit(X_tr, noisy(p)).score(X_tr, noisy(p)) for p in ps})
print(pd.DataFrame({"optimal k": optimal_k, "optimal acc": optimal_acc, "acc with k=1": k1_acc, "train with k=1": k1_train_acc}).round(3))

In [ ]:
cv_k, acc_with_cv_k = {}, {}
for p in ps:
    search = GridSearchCV(KNeighborsClassifier(), {"n_neighbors": Ks}, cv=5).fit(X_tr, noisy(p))
    cv_k[p] = search.best_params_["n_neighbors"]
    acc_with_cv_k[p] = search.score(X_te, y_te)
cv_k, acc_with_cv_k = pd.Series(cv_k), pd.Series(acc_with_cv_k)
print(pd.DataFrame({"k chosen in cv": cv_k, "test accuracy": acc_with_cv_k}).round(3))

## Check

In [ ]:
with verify():
    check(acc.shape == (5, 10) and acc.notna().all().all(), "complete 5 x 10 table", "one accuracy for every pair (p, k)")
    check(optimal_k.is_monotonic_increasing and optimal_k[0] <= 9 and optimal_k[0.4] >= 61, "the optimal k grows with the noise: from 9 or less (p = 0) to 61 or more (p = 0.4)", "argmax per row of acc")
    check(optimal_acc.min() > 0.97, "with the right k the accuracy stays above 0.97 even with 40% wrong labels", "optimal_acc = maximum per row")
    check(k1_acc[0.4] < 0.7 and k1_acc[0] > 0.97 and k1_acc[0.4] < k1_acc[0.2] < k1_acc[0.1] < k1_acc[0], "with k = 1 the accuracy keeps falling, down to below 0.7", "the k = 1 column of acc")
    check((k1_train_acc == 1.0).all(), "with k = 1 the training accuracy (on noisy labels) is always 1.0: it memorizes the errors too", "the nearest neighbor of a training point is the point itself")
    check(cv_k[0.3] >= 25 and cv_k[0.4] >= 25 and (acc_with_cv_k > 0.97).all(), "cross-validation on the noisy training set finds large k (25 or more with 30-40% noise) and the test accuracy stays above 0.97", "GridSearchCV(KNeighborsClassifier(), {'n_neighbors': Ks}, cv=5)")
done()

## What you learned

- **With `k = 1` the training accuracy is always 1.0**, and with 40% wrong labels it means the model has memorized all the errors. Measured on the clean test set the same choice is worth 0.99 without noise and 0.63 with 40%.
  A training accuracy of 1.0 says nothing.
- The **optimal `k` grows with the noise** (from 5 to 15, 41 and 121): every time the labels are less reliable it is better to vote among more neighbors, so the errors compensate. It is the bias-variance tradeoff in pure form: high noise = high variance
  = you need a smoother model.
- With the right `k` the test accuracy stays at 0.99 **even with 40% wrong labels**: KNN (with enough data and a problem with only two dimensions) is very robust to symmetric label noise, as long as `k` is chosen well.
- Cross-validation on the **noisy** training set still finds large `k`: the validation folds have the same noise, so the models that memorize the noise get penalized. You don't need clean labels to choose `k`, but you do need
  many examples: with little data and a lot of noise the choice becomes unstable.

## Common pitfall

Using `k = 1` or `k = 3` as a "default" without validating. With noisy labels (frequent in real data: human annotations, legacy systems) they are the worst choices. And never trust the training accuracy of a KNN.

## Interview question

<details><summary>You suspect that 20% of the labels are wrong. What do you change in a KNN?</summary>

I increase `k` (and choose it in cross-validation, which sees the same noise) to compensate for the errors by voting among more neighbors; I use 1/distance weights with caution because a wrong neighbor very close weighs too much. If I can, I clean up: methods like
`k=1` with a filter (edited nearest neighbours) or the search for labels inconsistent with the neighbors. And I always measure on a test set with verified labels, otherwise I measure the noise too.

</details>